In [75]:
import requests
import re
import json
import csv

In [76]:
url="https://www.stage2.capital/team?type=Catalyst+LP&function=*&industry=*"
response=requests.get(url)
html=response.text
print(f"Fetched{len(html)} characters")

Fetched4967420 characters


In [77]:
pattern=r'data-column-properties="({[^"]+})"'
matches=re.findall(pattern,html)
print(f"Fetched{len(matches)} data entries")

Fetched1000 data entries


In [78]:
lps=[]
for match in matches:
    json_str=match.replace('&quot;','"')
    data=json.loads(json_str)

    if data.get('type')=='Catalyst LP':
        lps.append({
            'name':data.get('name'),
            'title':data.get('title'),
            'company':data.get('company')
        })
print(f"Total Catalyst LPs: {len(lps)}")

Total Catalyst LPs: 242


In [79]:
def is_svp_plus(title):
    if not title:
        return False
    title_lower = title.lower().strip()


    if any(word in title_lower for word in ['fractional', 'consultant', 'advisor', 'interim']):
        return False


    if 'vice president' in title_lower:
        is_qualified_vp = (
            re.search(r'\bsenior\s+vice\s+president\b', title_lower) or
            re.search(r'\bexecutive\s+vice\s+president\b', title_lower) or
            re.search(r'\bgroup\s+vice\s+president\b', title_lower) or
            re.search(r'\b(global|regional|area|corporate)\s+.*vice\s+president\b', title_lower)
        )
        if is_qualified_vp:
            return True
        else:
            return False


    if re.search(r'\bchief\b.*\bofficer\b', title_lower):
        return True


    c_level_abbrevs = ['ceo', 'cro', 'cmo', 'cfo', 'coo', 'cto', 'cco', 'cio']
    for abbrev in c_level_abbrevs:
        if re.search(rf'\b{abbrev}\b', title_lower):
            return True

    if re.search(r'\b(svp|evp|gvp)\b', title_lower):
        return True


    if 'president' in title_lower:
        return True

    return False

In [80]:
def extract_catalyst_lps():

    print("Fetching Catalyst LPs page from Stage 2 Capital...")

    url = "https://www.stage2.capital/team?type=Catalyst+LP&function=*&industry=*"
    headers = {'User-Agent': 'Mozilla/5.0'}

    response = requests.get(url, headers=headers, timeout=30)
    html = response.text

    print(f"Successfully fetched page (length: {len(html):,} characters)")


    pattern = r'data-column-properties="({[^"]+})"'
    matches = re.findall(pattern, html)

    print(f"Found {len(matches)} profile data entries")

    lps = []

    for match in matches:
        try:

            json_str = match.replace('&quot;', '"')
            data = json.loads(json_str)


            if data.get('type') == 'Catalyst LP':
                name = (data.get('name') or '').strip()
                title = (data.get('title') or '').strip()
                company = (data.get('company') or '').strip()

                if name and title:
                    name_slug = name.lower().replace(' ', '-').replace('.', '')
                    profile_url = f"https://www.stage2.capital/team/{name_slug}"

                    lps.append({
                        'name': name,
                        'title': title,
                        'company': company,
                        'profile_url': profile_url
                    })

        except json.JSONDecodeError:
            continue

    print(f"\nTotal Catalyst LPs found: {len(lps)}")


    svp_plus_lps = []
    for lp in lps:
        if is_svp_plus(lp['title']):
            svp_plus_lps.append(lp)

    print(f"SVP+ Catalyst LPs: {len(svp_plus_lps)}")


    unique_lps = []
    seen_names = set()

    for lp in svp_plus_lps:
        if lp['name'] not in seen_names:
            seen_names.add(lp['name'])
            unique_lps.append(lp)

    svp_plus_lps = unique_lps
    print(f"After deduplication: {len(svp_plus_lps)}")


    svp_plus_lps.sort(key=lambda x: x['name'])

    return svp_plus_lps

In [81]:
test_titles=["Founder","Regional Vice President","Chief Revenue Officer"]
for title in test_titles:
    print(f"{title}: {is_svp_plus(title)}")

Founder: False
Regional Vice President: True
Chief Revenue Officer: True


In [82]:
svp_plus_lps = []
for lp in lps:
    if is_svp_plus(lp['title']):
        svp_plus_lps.append(lp)

print(f"SVP+ Catalyst LPs: {len(svp_plus_lps)}")
# Remove duplicates
unique_lps = []
seen_names = set()

for lp in svp_plus_lps:
    if lp['name'] not in seen_names:
        seen_names.add(lp['name'])
        unique_lps.append(lp)

svp_plus_lps = unique_lps
print(f"After deduplication: {len(svp_plus_lps)}")

SVP+ Catalyst LPs: 86
After deduplication: 85


In [83]:
svp_plus_lps = extract_catalyst_lps()
os.makedirs('outputs', exist_ok=True)

with open('outputs/catalyst_lps_svp_plus.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['name', 'title', 'company', 'profile_url'])
    writer.writeheader()
    writer.writerows(svp_plus_lps)

print(f"Saved {len(svp_plus_lps)} Catalyst LPs to outputs/catalyst_lps_svp_plus.csv")

Fetching Catalyst LPs page from Stage 2 Capital...
Successfully fetched page (length: 4,967,420 characters)
Found 1000 profile data entries

Total Catalyst LPs found: 226
SVP+ Catalyst LPs: 86
After deduplication: 85
Saved 85 Catalyst LPs to outputs/catalyst_lps_svp_plus.csv


In [84]:
# Add profile URLs
for lp in svp_plus_lps:
    name_slug = lp['name'].lower().replace(' ', '-').replace('.', '')
    lp['profile_url'] = f"https://www.stage2.capital/team/{name_slug}"

# Sort by name
svp_plus_lps.sort(key=lambda x: x['name'])

# Save final version
with open('outputs/catalyst_lps_svp_plus_final.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['name', 'title', 'company', 'profile_url'])
    writer.writeheader()
    writer.writerows(svp_plus_lps)

print("Final file ready: outputs/catalyst_lps_svp_plus_final.csv")

Final file ready: outputs/catalyst_lps_svp_plus_final.csv
